**Cell 1**

# NB13 — Methodenvergleich auf den HelpSteer2-DPO-Adaptern aus NB11

Dieses Notebook überträgt das zweiphasige NB10-Protokoll auf **genau die fünf
eingefrorenen DPO-Experten aus NB11**. Es berechnet zuerst deren eigene
Update-Geometrie und vergleicht anschließend Baseline, Avg, Cert, MaxMin($c$)
und Fair($\alpha,\varepsilon$).

Die Baseline $\lambda=p$ ist der präferenzgewichtete Merge der fünf
DPO-Updates, nicht das unveränderte Basismodell. Für jede Methode gilt

$$\theta(\lambda)=\theta_0+\sum_i\lambda_i\Delta_i^{\mathrm{DPO}}.$$

| Phase | Inhalt | ArmoRM | Standardmäßig aktiv? |
|---|---|---:|---:|
| **A** | Manifeste prüfen, $R_{\mathrm{DPO}}$ berechnen, Koeffizienten und Diagnostik erzeugen | nein | ja |
| **B** | neue, eingefrorene Promptmenge generieren, alle deduplizierten Merges bewerten | nur post-hoc | nein |

ArmoRM wurde laut NB11-Manifesten weder zur Paarbildung noch zum Training oder
zur Checkpointwahl verwendet. Die 64 Prompts aus der NB11-Differenzierungsdiagnose
sind jedoch bereits verbraucht; NB13 rekonstruiert und sperrt sie anhand des
gepinnten Datensatz-Commits und ihres bekannten Hashes.

Die primäre Messgröße ist der rohe präferenzgewichtete ArmoRM-Score
$U_p(\lambda)=p^\top r(\lambda)$. Die Proxy-Geometrie $R$ wird nicht mit der
externen Messung gleichgesetzt. Korrelationen zwischen Proxy und ArmoRM werden
separat und deskriptiv ausgewiesen.

**Wichtig:** NB13 übernimmt keine Matrix, kein Zertifikat, keinen Reward-Cache
und keine Schlussfolgerung aus NB10/NB09.1. Insbesondere wird kein pauschales
Äquikorrelation-Kriterium verwendet; Floor-Aussagen stammen ausschließlich aus
den auf $R_{\mathrm{DPO}}$ gelösten LPs.


**Cell 2**

## 1. Repository und NB11-Adapterbundle


In [ ]:
# Cell 3
%cd /content
import hashlib, os, stat, shutil, zipfile
from pathlib import Path

repo_path = "/content/master-thesis"
repo_url = "https://github.com/NZhang137/master-thesis.git"

if os.path.isdir(os.path.join(repo_path, ".git")):
    %cd /content/master-thesis
    !git pull
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {repo_url} {repo_path}
    %cd /content/master-thesis

# This notebook is bound to the exact bundle supplied after NB11. Files in
# the archive are treated as data; only the five manifests/configs/weights
# listed below are extracted.
EXPECTED_ADAPTER_BUNDLE_SHA256 = "97f41c1f2290bd1074a261577de5c273c3754d89e311f713d90d7d7cc2a5bbf3"
ADAPTER_AXES = ("helpfulness", "correctness", "coherence", "complexity", "verbosity")
ADAPTER_INPUT_ROOT = Path(repo_path) / "results/dpo_rq2/nb11_pairs2690_seed137_run1"
REQUIRED_ARCHIVE_MEMBERS = {
    f"dpo_{axis}/{relative}"
    for axis in ADAPTER_AXES
    for relative in (
        "training_manifest.json",
        "adapter/adapter_config.json",
        "adapter/adapter_model.safetensors",
    )
}

candidates = sorted({
    *Path("/content").glob("nb11_tinyllama_helpsteer2_dpo_adapters*.zip"),
    *Path.cwd().glob("nb11_tinyllama_helpsteer2_dpo_adapters*.zip"),
})
if not candidates:
    from google.colab import files

    print("Bitte jetzt nb11_tinyllama_helpsteer2_dpo_adapters.zip auswählen.")
    uploaded = files.upload()
    candidates = [Path.cwd() / name for name in uploaded if name.lower().endswith(".zip")]

matching = []
observed_hashes = {}
for candidate in candidates:
    digest = hashlib.sha256(candidate.read_bytes()).hexdigest()
    observed_hashes[str(candidate)] = digest
    if digest == EXPECTED_ADAPTER_BUNDLE_SHA256:
        matching.append(candidate.resolve())
if len(matching) != 1:
    raise RuntimeError(
        "Expected exactly one ZIP with the frozen NB11 SHA256. Observed: "
        + repr(observed_hashes)
    )

ADAPTER_BUNDLE_PATH = matching[0]
ADAPTER_BUNDLE_SHA256 = EXPECTED_ADAPTER_BUNDLE_SHA256
required_on_disk = [ADAPTER_INPUT_ROOT / name for name in sorted(REQUIRED_ARCHIVE_MEMBERS)]
if not all(path.is_file() for path in required_on_disk):
    with zipfile.ZipFile(ADAPTER_BUNDLE_PATH) as bundle:
        entries = {entry.filename: entry for entry in bundle.infolist()}
        missing = sorted(REQUIRED_ARCHIVE_MEMBERS - set(entries))
        if missing:
            raise FileNotFoundError(f"Adapter ZIP is incomplete; missing: {missing}")
        total_size = sum(entries[name].file_size for name in REQUIRED_ARCHIVE_MEMBERS)
        if total_size > 750_000_000:
            raise RuntimeError(f"Refusing unexpectedly large adapter payload: {total_size} bytes")
        target = ADAPTER_INPUT_ROOT.resolve()
        target.mkdir(parents=True, exist_ok=True)
        for name in sorted(REQUIRED_ARCHIVE_MEMBERS):
            entry = entries[name]
            unix_mode = (entry.external_attr >> 16) & 0o170000
            if unix_mode == stat.S_IFLNK:
                raise RuntimeError(f"Symlink forbidden in adapter ZIP: {name}")
            destination = (target / name).resolve()
            if target not in destination.parents:
                raise RuntimeError(f"Unsafe path in adapter ZIP: {name}")
            destination.parent.mkdir(parents=True, exist_ok=True)
            with bundle.open(entry) as source, destination.open("wb") as output:
                shutil.copyfileobj(source, output)

missing = [str(path) for path in required_on_disk if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Extraction incomplete: {missing}")
print(f"[OK] Frozen NB11 bundle: {ADAPTER_BUNDLE_SHA256}")
print(f"[OK] Required files for {len(ADAPTER_AXES)} DPO experts are available.")


**Cell 4**

## 2. Runtime und Abhängigkeiten


In [ ]:
# Cell 5
!nvidia-smi || echo "Keine GPU — Phase A läuft trotzdem."


In [ ]:
# Cell 6
# The inference stack matches NB11's adapter-compatible Transformers/PEFT
# versions. Phase B additionally uses bitsandbytes for the post-hoc evaluator.
import importlib.metadata, subprocess, sys

print(f"Python {sys.version.split()[0]} ({sys.executable})")
packages = [
    "pandas==2.2.3", "numpy==2.1.3", "scipy==1.14.1",
    "protobuf==5.29.5", "transformers==4.45.2", "tokenizers==0.20.3",
    "peft==0.13.2", "accelerate==1.1.1", "datasets==3.1.0",
    "huggingface_hub==0.36.0", "bitsandbytes", "pyyaml", "safetensors", "psutil",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "--only-binary=:all:", *packages],
    check=True,
)
expected = {"transformers": "4.45.2", "tokenizers": "0.20.3", "peft": "0.13.2"}
installed = {name: importlib.metadata.version(name) for name in expected}
assert installed == expected, f"Unexpected inference stack: {installed}"
print("Runtime:", {name: importlib.metadata.version(name) for name in
                   ("transformers", "tokenizers", "peft", "accelerate", "datasets")})


**Cell 7**

## 3. Einstellungen


In [ ]:
# Cell 8
from __future__ import annotations

import gc, hashlib, json, re, sys, zipfile
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("/content/master-thesis").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RUN_TAG = "nb13_nb11_dpo_run1"
CLAIM_STATUS = "exploratory"  # change only before freezing; never after reward contact
CONFIG_PATH = PROJECT_ROOT / "configs/tinyllama_helpsteer2_armorm.yaml"
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks/13_method_comparison_helpsteer2_dpo_colab.ipynb"

RESULTS_DIR = PROJECT_ROOT / "results/nb13_helpsteer2_dpo_method_comparison" / RUN_TAG
LAMBDA_CSV = RESULTS_DIR / "lambda_table.csv"
PREREG_JSON = RESULTS_DIR / "nb13_preregistration.json"
REWARD_CACHE = RESULTS_DIR / "reward_cache.jsonl"
REWARD_PROMPT_PATH = RESULTS_DIR / "nb13_reward_prompts.jsonl"
DIAGNOSTIC_EXCLUSION_PATH = RESULTS_DIR / "nb11_diagnostic_prompts_exclusion.jsonl"
COSINE_MATRIX_CSV = RESULTS_DIR / "R_dpo_cos.csv"
GRAM_MATRIX_CSV = RESULTS_DIR / "R_dpo_gram.csv"
D_NORMS_CSV = RESULTS_DIR / "dpo_update_norms.csv"
GEOMETRY_JSON = RESULTS_DIR / "dpo_geometry_report.json"
FINAL_CSV = RESULTS_DIR / "method_comparison.csv"
ROBUST_CSV = RESULTS_DIR / "normalization_robustness.csv"
STATS_CSV = RESULTS_DIR / "paired_statistics.csv"
PROXY_CSV = RESULTS_DIR / "proxy_method_diagnostics.csv"
PROXY_JSON = RESULTS_DIR / "proxy_validation_report.json"
REPORT_JSON = RESULTS_DIR / "nb13_report.json"
REWARD_TENSOR_PATH = RESULTS_DIR / "nb13_reward_tensor.npy"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_MATRIX = "R_cos"
RHO_GRID = [0.0, 0.1, 0.2, 0.5]
CERT_C = 0.5
CERT_EPS = 1e-8
C_GRID = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 1.0]
ALPHA_GRID = [0.0, 0.5, 1.0, 2.0]
EPS_GRID = [0.01, 0.02, 0.05, 0.10]
DIRICHLET_SEED = 137

# Phase B is deliberately off. Freeze and inspect Phase A first, then set
# PREREG_CONFIRM=True; only after that set RUN_REWARD_COLLECTION=True.
PREREG_CONFIRM = False
RUN_REWARD_COLLECTION = False
N_EVAL_PROMPTS = 80
EVAL_PROMPT_SEED = 1313
MAX_NEW_TOKENS = 256
REPETITION_PENALTY = 1.15
NO_REPEAT_NGRAM_SIZE = 5
LAMBDA_DEDUP_DECIMALS = 8

ARMORM_MODEL = "RLHFlow/ArmoRM-Llama3-8B-v0.1"
ARMORM_REVISION = "eb2676d20da2f2d41082289d23c59b9f7427f955"
ARMORM_PRECISION = "8bit"  # matches NB11's diagnostic precision; not a training reward
NB11_DIAGNOSTIC_PROMPT_SHA256 = "3864d363dbff91fdeda62029a16b0da3b3436fe3557d42cb75739c75649ff51e"

def _sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

print(f"RUN_TAG      = {RUN_TAG}")
print(f"claim status = {CLAIM_STATUS}")
print(f"results      = {RESULTS_DIR}")
print(f"Phase B      = {'ACTIVE' if RUN_REWARD_COLLECTION else 'off'}")


**Cell 9**

## 4. Kanonische Implementierungen aus `src/`

Die Methoden werden hier nicht neu definiert. NB13 importiert dieselben
kanonischen Implementierungen wie NB10; geändert werden ausschließlich
Adapterregime, Geometrie, Promptbindung und Evaluation.

| Label | Implementierung |
|---|---|
| Avg | `src.coefficient_portfolio.avg` |
| Cert | `src.coefficient_portfolio.cert` |
| MaxMin | `src.coefficient_portfolio.maxmin_c` |
| Fair | `src.coefficient_portfolio.fair_alpha_eps` |


In [ ]:
# Cell 10
from src.experiment_config import get_attribute_order, load_experiment_config, validate_preference_vectors
from src.proxy_validation import (
    build_search_set,
    coefficient_key,
    normalization_agreement,
    preference_utility,
    run_spearman_analysis,
    safe_spearman,
    write_json,
)
from src.coefficient_portfolio import (
    avg,
    cert,
    fair_alpha_eps,
    floor_lp_at_p,
    improvements,
    maxmin_c,
)
from src.lambda_utils import lambda_key

print("Portfolio imported; no coefficient method is defined in NB13.")


**Cell 11**

## 5. Konfiguration und Präferenzen


In [ ]:
# Cell 12
config = load_experiment_config(CONFIG_PATH)
ATTRIBUTES = get_attribute_order(config)
PREFERENCES = validate_preference_vectors(config)
m = len(ATTRIBUTES)
assert tuple(ATTRIBUTES) == tuple(ADAPTER_AXES)
assert Path(ADAPTER_INPUT_ROOT).is_dir()

R_cos = R_gram = R = None
eigenvalues = None
print(f"Attribute order: {list(ATTRIBUTES)}")


**Cell 13**

## 6. NB11-Provenienz und DPO-Geometrie

Diese Zelle verifiziert jedes Trainingsmanifest und jeden Adapterhash. Danach
berechnet sie Gram- und Kosinusmatrix direkt aus den effektiven LoRA-Updates
$\Delta_i=s_iB_iA_i$. Es wird keine Matrix aus einem anderen Regime geladen.

Die Low-Rank-Identität für Frobenius-Produkte wird zusätzlich auf zwei
deterministisch gewählten Layern gegen explizit materialisierte Updates
geprüft. Das ist ein numerischer Pfadtest, ohne alle fünf Vollmatrizen im RAM
zu halten.


In [ ]:
# Cell 14
import math
import torch
from src.effective_lora_geometry import (
    effective_lora_inner_product,
    effective_lora_update_norm,
    effective_lora_update_numel,
    load_effective_lora_geometry,
    validate_compatible_geometries,
)
from src.preferences import PREFERENCES as PREFERENCES_MODULE

REGIME = "helpsteer2_dpo_nb11"
EXPECTED_BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
EXPECTED_BASE_REVISION = "fe8a4ea1ffedaf415f4da2f062534de366a451e6"
EXPECTED_DATASET = "nvidia/HelpSteer2"
EXPECTED_DATASET_REVISION = "990b2711a36180dd19d9c94b8627844866f8982a"

# Preferences remain a single-source consistency check.
assert set(PREFERENCES) == set(PREFERENCES_MODULE)
for name in PREFERENCES:
    assert np.allclose(PREFERENCES[name], PREFERENCES_MODULE[name], atol=1e-12)

ADAPTER_PATHS = {
    axis: (ADAPTER_INPUT_ROOT / f"dpo_{axis}/adapter").resolve()
    for axis in ATTRIBUTES
}
MANIFEST_PATHS = {
    axis: (ADAPTER_INPUT_ROOT / f"dpo_{axis}/training_manifest.json").resolve()
    for axis in ATTRIBUTES
}
MANIFESTS = {}
for axis in ATTRIBUTES:
    manifest = json.loads(MANIFEST_PATHS[axis].read_text(encoding="utf-8"))
    weights_path = ADAPTER_PATHS[axis] / "adapter_model.safetensors"
    assert manifest["completed"] is True
    assert manifest["training_method"] == "DPO"
    assert manifest["armorm_used_during_training"] is False
    assert manifest["checkpoint_selection"] == "fixed final epoch; no reward-model selection"
    assert manifest["reward_name"] == axis
    assert manifest["base_model_name"] == EXPECTED_BASE_MODEL
    assert manifest["base_revision"] == EXPECTED_BASE_REVISION
    assert manifest["dataset_name"] == EXPECTED_DATASET
    assert manifest["dataset_revision"] == EXPECTED_DATASET_REVISION
    assert manifest["dataset_split"] == "train"
    assert manifest["selected_pair_count"] == 2690
    assert manifest["adapter_model_sha256"] == _sha256_file(weights_path)
    MANIFESTS[axis] = manifest
    print(f"[OK] {axis:12s} weights={manifest['adapter_model_sha256'][:16]}…")

shared_fields = (
    "trainer_script_sha256", "runtime_versions", "base_model_name", "base_revision",
    "dataset_name", "dataset_revision", "dataset_fingerprint", "dataset_split",
    "selected_pair_count", "beta", "dpo_disable_dropout", "epochs",
    "learning_rate", "effective_batch_size", "max_length", "max_prompt_length",
    "precision", "seed", "pair_seed", "lora",
)
for field in shared_fields:
    values = {json.dumps(MANIFESTS[axis][field], sort_keys=True) for axis in ATTRIBUTES}
    assert len(values) == 1, f"NB11 experts differ on {field}: {values}"

BASE_MODEL_NAME = MANIFESTS[ATTRIBUTES[0]]["base_model_name"]
BASE_REVISION = MANIFESTS[ATTRIBUTES[0]]["base_revision"]
DATASET_NAME = MANIFESTS[ATTRIBUTES[0]]["dataset_name"]
DATASET_REVISION = MANIFESTS[ATTRIBUTES[0]]["dataset_revision"]
assert config["base_model_name"] == BASE_MODEL_NAME
assert config["dataset_name"] == DATASET_NAME

geometries = {
    axis: load_effective_lora_geometry(ADAPTER_PATHS[axis])
    for axis in ATTRIBUTES
}
validate_compatible_geometries(
    [geometries[axis] for axis in ATTRIBUTES], list(ATTRIBUTES)
)

R_gram = np.empty((m, m), dtype=np.float64)
for i, left in enumerate(ATTRIBUTES):
    for j, right in enumerate(ATTRIBUTES[: i + 1]):
        value = effective_lora_inner_product(geometries[left], geometries[right])
        R_gram[i, j] = R_gram[j, i] = value
d_norms = np.sqrt(np.maximum(np.diag(R_gram), 0.0))
for i, axis in enumerate(ATTRIBUTES):
    assert np.isclose(d_norms[i], effective_lora_update_norm(geometries[axis]), rtol=1e-10)
R_cos = R_gram / np.outer(d_norms, d_norms)
R_cos = 0.5 * (R_cos + R_cos.T)

# Explicit cross-check of the rank-space identity on the two smallest layers.
reference_geometry = geometries[ATTRIBUTES[0]]
crosscheck_modules = sorted(
    reference_geometry,
    key=lambda name: math.prod(reference_geometry[name].effective_shape),
)[:2]
crosscheck_errors = []
for module_name in crosscheck_modules:
    materialized = {}
    for axis in ATTRIBUTES:
        layer = geometries[axis][module_name]
        materialized[axis] = layer.scaling * (layer.lora_b @ layer.lora_a)
    for left in ATTRIBUTES:
        for right in ATTRIBUTES:
            direct = float(torch.sum(materialized[left] * materialized[right]).item())
            a = geometries[left][module_name]
            b = geometries[right][module_name]
            low_rank = float(
                a.scaling * b.scaling
                * torch.sum((a.lora_b.T @ b.lora_b) * (a.lora_a @ b.lora_a.T)).item()
            )
            crosscheck_errors.append(abs(direct - low_rank) / max(abs(direct), 1.0))
    del materialized
max_crosscheck_error = float(max(crosscheck_errors, default=0.0))
assert max_crosscheck_error < 1e-10

assert np.allclose(R_cos, R_cos.T, atol=1e-12)
assert np.allclose(np.diag(R_cos), 1.0, atol=1e-10)
eigenvalues = np.linalg.eigvalsh(R_cos)
assert eigenvalues.min() > 1e-10 * eigenvalues.max(), (
    f"R_dpo_cos is not numerically positive definite: {eigenvalues}"
)
R = R_cos if PRIMARY_MATRIX == "R_cos" else R_gram

pd.DataFrame(R_cos, index=ATTRIBUTES, columns=ATTRIBUTES).to_csv(COSINE_MATRIX_CSV)
pd.DataFrame(R_gram, index=ATTRIBUTES, columns=ATTRIBUTES).to_csv(GRAM_MATRIX_CSV)
pd.DataFrame({"attribute": ATTRIBUTES, "effective_update_norm": d_norms}).to_csv(
    D_NORMS_CSV, index=False
)

inverse_ones = np.linalg.solve(R, np.ones(m))
offdiag = R_cos[np.triu_indices(m, 1)]
GEOMETRY_SUMMARY = {
    "regime": REGIME,
    "primary_matrix": PRIMARY_MATRIX,
    "attribute_order": list(ATTRIBUTES),
    "effective_update_numel": int(effective_lora_update_numel(reference_geometry)),
    "update_norms": dict(zip(ATTRIBUTES, map(float, d_norms))),
    "cosine_eigenvalues": eigenvalues.tolist(),
    "cosine_condition_number": float(eigenvalues.max() / eigenvalues.min()),
    "cosine_dominant_eigenvalue_share": float(eigenvalues.max() / eigenvalues.sum()),
    "cosine_offdiag_min": float(offdiag.min()),
    "cosine_offdiag_max": float(offdiag.max()),
    "cosine_offdiag_mean": float(offdiag.mean()),
    "inverse_R_ones": inverse_ones.tolist(),
    "inverse_R_ones_strictly_positive": bool(np.all(inverse_ones > 1e-10)),
    "materialized_crosscheck_modules": crosscheck_modules,
    "materialized_crosscheck_max_relative_error": max_crosscheck_error,
    "matrix_sha256": {
        "R_cos": _sha256_file(COSINE_MATRIX_CSV),
        "R_gram": _sha256_file(GRAM_MATRIX_CSV),
    },
}
write_json(GEOMETRY_JSON, GEOMETRY_SUMMARY)
print(json.dumps(GEOMETRY_SUMMARY, indent=2))
display(pd.DataFrame(R_cos, index=ATTRIBUTES, columns=ATTRIBUTES).round(4))


**Cell 15**

### 6.1 Präferenzmenge

Phase A nutzt die 11 benannten Präferenzen aus der zentralen Konfiguration
plus 64 deterministische Dirichlet-Punkte (`seed=137`, $\alpha=1$,
`min_component=0.02`). Phase B bleibt auf die 11 benannten Präferenzen
beschränkt. Diese Auswahl wird vor jedem ArmoRM-Kontakt gebunden.


In [ ]:
# Cell 16
USE_FULL_SEARCH_SET = True
PREF_SET = [(name, np.asarray(vec, dtype=np.float64)) for name, vec in PREFERENCES.items()]

if USE_FULL_SEARCH_SET:
    seen = {tuple(np.round(v, 9)) for _, v in PREF_SET}
    draws = build_search_set(
        m,
        n_dirichlet=64,
        dirichlet_alpha=1.0,
        preferences=list(PREFERENCES.values()),
        seed=DIRICHLET_SEED,
    )
    for index, vector in enumerate(draws):
        key = tuple(np.round(vector, 9))
        if key not in seen:
            seen.add(key)
            PREF_SET.append((f"dirichlet_{index:02d}", np.asarray(vector, dtype=np.float64)))

for name, p in PREF_SET:
    assert abs(float(p.sum()) - 1.0) < 1e-9 and np.all(p >= -1e-12), name
INTERIOR = [name for name, p in PREF_SET if np.all(p > 1e-12)]
print(f"Phase-A preferences: {len(PREF_SET)} ({len(INTERIOR)} interior)")
display(pd.DataFrame([p for _, p in PREF_SET], index=[n for n, _ in PREF_SET],
                     columns=list(ATTRIBUTES)).round(4))


**Cell 17**

### 6.2 Phase-B-Teilmenge

Die Auswahlregel wird ohne Rewardwerte festgelegt: alle benannten
Präferenzen und alle dort gültigen Methodenkandidaten. Identische
Koeffizienten werden erst bei der Auswertung dedupliziert.


In [ ]:
# Cell 18
PHASE_B_RULE = "all 11 named preferences; all usable frozen-grid candidates; exact lambda deduplication"
PREF_SET_B = [(name, p) for name, p in PREF_SET if name in PREFERENCES]
PHASE_B_NAMES = {name for name, _ in PREF_SET_B}
assert len(PREF_SET_B) == len(PREFERENCES)
print(f"Phase A: |P|={len(PREF_SET)}; Phase B: |P_B|={len(PREF_SET_B)}")


**Cell 19**

## 7. Phase A — alle Koeffizienten berechnen (rewardfrei)


In [ ]:
# Cell 20
def norm_R(v):
    return float(np.sqrt(max(v @ R @ v, 0.0)))


rows = []
for pname, p in PREF_SET:
    base = {"p_name": pname, **{f"p_{a}": float(p[i]) for i, a in enumerate(ATTRIBUTES)}}

    results = []

    for rho in RHO_GRID:
        lam = avg(p, R, rho)
        results.append(("Avg", f"rho={rho}", lam, "OK", {}))

    lam_cert, t_cert = cert(p, R, CERT_C, CERT_EPS)
    t_lp, collapsed = floor_lp_at_p(R, p)
    results.append(("Cert", "", lam_cert,
                    "FLOOR_COLLAPSED" if collapsed else "FLOOR_NONTRIVIAL",
                    {"t_star": t_cert, "t_star_lp": t_lp}))

    for c in C_GRID:
        lam_mm, t_mm, status = maxmin_c(p, R, c)
        results.append(("MaxMin", f"c={c}", lam_mm, status, {"t_star": t_mm}))

    for alpha in ALPHA_GRID:
        for eps in EPS_GRID:
            lam_f, u_f, status = fair_alpha_eps(p, R, alpha, eps)
            results.append(("Fair", f"alpha={alpha},eps={eps}", lam_f, status, {"u_alpha": u_f}))

    for method, params, lam, status, extra in results:
        row = dict(base, method=method, params=params, status=status, **extra)
        if lam is None:
            row.update({f"lam_{a}": np.nan for a in ATTRIBUTES})
            row.update({"dist_l2": np.nan, "dist_R": np.nan, "proxy_pRlam": np.nan,
                        "min_delta": np.nan, "moved": False, "usable": False})
        else:
            v = lam - p
            row.update({f"lam_{a}": float(lam[i]) for i, a in enumerate(ATTRIBUTES)})
            row.update({"dist_l2": float(np.linalg.norm(v)),
                        "dist_R": norm_R(v),
                        "proxy_pRlam": float(p @ R @ lam),
                        "min_delta": float(np.min(improvements(p, R, lam))),
                        "moved": bool(np.linalg.norm(v) > 1e-8),
                        "usable": True})
        rows.append(row)

lam_df = pd.DataFrame(rows)
lam_df.to_csv(LAMBDA_CSV, index=False)
lam_cols = [f"lam_{a}" for a in ATTRIBUTES]
p_cols = [f"p_{a}" for a in ATTRIBUTES]
print(f"{len(lam_df)} lambda rows -> {LAMBDA_CSV}")
display(lam_df.head(20))

**Cell 21**

### 7.1 Bewegungsdiagnostik

`moved=False` bezeichnet einen gültigen Kollaps auf $\lambda=p$.
`INFEASIBLE_BALL_MISSES_SIMPLEX` bezeichnet dagegen eine leere zulässige
Menge. Beide Fälle bleiben getrennt.


In [ ]:
# Cell 22
key_series = lam_df["method"] + lam_df["params"].map(lambda s: f"({s})" if s else "")
summary = (lam_df.assign(key=key_series).groupby("key")
           .agg(n=("moved", "size"),
                n_usable=("usable", "sum"),
                n_moved=("moved", "sum"),
                mean_dist_l2=("dist_l2", "mean"),
                max_dist_l2=("dist_l2", "max"),
                mean_dist_R=("dist_R", "mean"))
           .sort_values("mean_dist_l2", ascending=False))
display(summary.round(4))

print("\nStatus distribution:")
display(lam_df["status"].value_counts().rename_axis("status").reset_index(name="n"))

unusable = lam_df[~lam_df["usable"]]
if len(unusable):
    print(f"\n{len(unusable)} rows without lambda:")
    display(unusable[["p_name", "method", "params", "status"]])
    n_infeasible = int((unusable["status"] == "INFEASIBLE_BALL_MISSES_SIMPLEX").sum())
    n_failed = len(unusable) - n_infeasible
    print(f"  including mathematically empty: {n_infeasible}   solver failures: {n_failed}")
    if n_failed:
        print("  WARNING: Solver failures are NOT a result and must be resolved before Phase B.")
else:
    print("\nAll rows have a usable lambda.")

**Cell 23**

### 7.2 Interne Konsistenz auf der neu berechneten DPO-Geometrie

Die folgenden Tests leiten ihre Erwartungen ausschließlich aus den aktuellen
DPO-LPs und Nebenbedingungen ab. Sie setzen weder einen Floor-Kollaps noch
$\lambda^{\mathrm{Cert}}=p$ im Voraus voraus.


In [ ]:
# Cell 24
TOL = 2e-6

# Every usable output must be a simplex vector.
usable_rows = lam_df[lam_df["usable"]]
usable_lambdas = usable_rows[lam_cols].to_numpy(float)
assert np.all(np.isfinite(usable_lambdas))
assert np.all(usable_lambdas >= -TOL)
assert np.allclose(usable_lambdas.sum(axis=1), 1.0, atol=TOL)

floor_by_preference = {}
for name, p in PREF_SET:
    t_lp, collapsed = floor_lp_at_p(R, p)
    floor_by_preference[name] = {"t_lp": float(t_lp), "collapsed": bool(collapsed)}

# Cert is checked against an independent floor LP for each p.
cert_rows = lam_df[lam_df["method"] == "Cert"]
for _, row in cert_rows.iterrows():
    floor = floor_by_preference[row["p_name"]]
    assert float(row["t_star"]) >= -TOL
    assert float(row["t_star"]) <= max(floor["t_lp"], 0.0) + TOL
    assert float(row["min_delta"]) >= -TOL
    if floor["collapsed"]:
        assert not bool(row["moved"]), row["p_name"]
n_collapsed = sum(entry["collapsed"] for entry in floor_by_preference.values())
print(f"[OK] Cert agrees with the p-aware floor LP; collapsed: {n_collapsed}/{len(PREF_SET)}")

# On a collapsed floor, MaxMin(c=1) must coincide with the baseline.
mm_one = lam_df[(lam_df["method"] == "MaxMin") & (lam_df["params"] == "c=1.0")]
for _, row in mm_one.iterrows():
    if floor_by_preference[row["p_name"]]["collapsed"]:
        assert bool(row["usable"]) and float(row["dist_l2"]) < TOL
print("[OK] MaxMin(c=1) respects every current floor-collapse certificate.")

# With a cosine matrix, a vertex's own diagonal is strictly larger than its
# off-diagonal similarities, so MaxMin's c<1 ball misses the simplex.
for name, p in PREF_SET:
    if np.count_nonzero(p > 1e-12) != 1:
        continue
    sub = lam_df[(lam_df["p_name"] == name) & (lam_df["method"] == "MaxMin")]
    for _, row in sub.iterrows():
        c_value = float(row["params"].split("=")[1])
        if c_value < 1.0 - 1e-9:
            assert row["status"] == "INFEASIBLE_BALL_MISSES_SIMPLEX"
print("[OK] Boundary infeasibility is explicit, not mislabeled as collapse.")

# A moving point on a collapsed floor must harm at least one proxy axis.
collapsed_names = {name for name, entry in floor_by_preference.items() if entry["collapsed"]}
collapsed_movers = lam_df[
    lam_df["p_name"].isin(collapsed_names) & lam_df["moved"] & lam_df["min_delta"].notna()
]
assert not (collapsed_movers["min_delta"] > TOL).any()

# Fair must honor its explicitly relaxed floor.
fair_rows = lam_df[(lam_df["method"] == "Fair") & lam_df["usable"]]
for _, row in fair_rows.iterrows():
    eps = float(row["params"].split("eps=")[1])
    assert float(row["min_delta"]) >= -eps - TOL
assert not (lam_df["status"] == "SOLVER_FAILED").any(), "Resolve solver failures before Phase B."
print("[OK] All usable points satisfy simplex/floor constraints; no solver failure remains.")

# G6: the notebook may orchestrate methods but may not redefine them.
if NOTEBOOK_PATH.is_file():
    notebook_text = NOTEBOOK_PATH.read_text(encoding="utf-8")
    forbidden = [r"def\s+avg\s*\(", r"def\s+cert\s*\(",
                 r"def\s+maxmin_c\s*\(", r"def\s+fair_alpha_eps\s*\("]
    hits = [pattern for pattern in forbidden if re.search(pattern, notebook_text)]
    assert not hits, f"Duplicate method definitions in NB13: {hits}"
    print("[OK] No coefficient method is redefined in the notebook.")


**Cell 25**

### 7.3 Phase-B-Budget

Jeder eindeutige Mergepunkt wird genau einmal bewertet. Punkte mit
$\lambda=p$ teilen den Cacheeintrag der Baseline; unzulässige Punkte werden
nicht generiert.


**Cell 26**

## 8. Neue Evaluationsprompts und Sperrliste

NB13 rekonstruiert zuerst exakt die 64 in NB11 verbrauchten
Differenzierungs-Prompts (`validation`, Seed 991). Der bekannte SHA256 muss
übereinstimmen. Anschließend werden 80 neue Prompts vom gepinnten
HelpSteer2-Commit gezogen und gegen NB06-, Projekt- und NB11-Prompts geprüft.


In [ ]:
# Cell 27
from datasets import load_dataset
from scripts.diff_experts import select_unique_prompts
from src.eval_prompts import build_eval_prompt_file, ensure_nb06_prompt_files

validation_rows = load_dataset(
    DATASET_NAME, split="validation", revision=DATASET_REVISION
)
nb11_diagnostic_prompts = select_unique_prompts(
    validation_rows, n_prompts=64, seed=991
)
prompt_payload = json.dumps(
    nb11_diagnostic_prompts, ensure_ascii=False, separators=(",", ":")
)
reconstructed_hash = hashlib.sha256(prompt_payload.encode("utf-8")).hexdigest()
assert reconstructed_hash == NB11_DIAGNOSTIC_PROMPT_SHA256, (
    "Could not reconstruct NB11's consumed diagnostic prompts: "
    f"{reconstructed_hash} != {NB11_DIAGNOSTIC_PROMPT_SHA256}"
)

exclusion_rows = [
    {
        "prompt_id": f"nb11_diagnostic_{index:03d}",
        "category": "nb11_posthoc_differentiation",
        "prompt": prompt,
        "notes": (
            f"Consumed by NB11 post-hoc differentiation; {DATASET_NAME} validation "
            f"at {DATASET_REVISION}, seed=991, prompt-list SHA256={reconstructed_hash}"
        ),
    }
    for index, prompt in enumerate(nb11_diagnostic_prompts, start=1)
]
if DIAGNOSTIC_EXCLUSION_PATH.exists():
    existing = [json.loads(line) for line in DIAGNOSTIC_EXCLUSION_PATH.read_text(
        encoding="utf-8").splitlines() if line.strip()]
    assert [row["prompt"] for row in existing] == nb11_diagnostic_prompts
else:
    with DIAGNOSTIC_EXCLUSION_PATH.open("w", encoding="utf-8") as handle:
        for row in exclusion_rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"[OK] Reconstructed and excluded all 64 NB11 diagnostic prompts: {reconstructed_hash}")

# Reconstruct the two historical NB06 sets deterministically when a fresh
# clone does not contain their result artifacts.
NB06_PROMPT_SUMMARY = ensure_nb06_prompt_files(
    PROJECT_ROOT,
    dataset_name=DATASET_NAME,
    dataset_revision=DATASET_REVISION,
    split="validation",
    seed=137,
    n_per_set=80,
)
PROMPT_SUMMARY = build_eval_prompt_file(
    REWARD_PROMPT_PATH,
    n=N_EVAL_PROMPTS,
    seed=EVAL_PROMPT_SEED,
    split="validation",
    dataset_name=DATASET_NAME,
    dataset_revision=DATASET_REVISION,
    prompt_id_prefix="nb13",
    project_root=PROJECT_ROOT,
    extra_exclude_paths=[DIAGNOSTIC_EXCLUSION_PATH],
    allow_missing_exclusions=False,
)
reward_prompt_rows = [json.loads(line) for line in REWARD_PROMPT_PATH.read_text(
    encoding="utf-8").splitlines() if line.strip()]
assert not ({row["prompt"] for row in reward_prompt_rows} & set(nb11_diagnostic_prompts))
assert PROMPT_SUMMARY["disjointness_verified"]
PROMPT_SUMMARY["prompt_file_sha256"] = _sha256_file(REWARD_PROMPT_PATH)
PROMPT_SUMMARY["nb11_diagnostic_prompt_list_sha256"] = reconstructed_hash
# Keep the frozen binding stable across first creation and later validation
# of the same file. Transient keys such as created/validated are reporting
# metadata, not scientific inputs.
PROMPT_BINDING = {
    "dataset_name": DATASET_NAME,
    "dataset_revision": DATASET_REVISION,
    "split": "validation",
    "seed": EVAL_PROMPT_SEED,
    "n": N_EVAL_PROMPTS,
    "prompt_file_sha256": _sha256_file(REWARD_PROMPT_PATH),
    "nb11_exclusion_file_sha256": _sha256_file(DIAGNOSTIC_EXCLUSION_PATH),
    "nb11_prompt_list_sha256": reconstructed_hash,
    "exclusion_files_missing": list(PROMPT_SUMMARY["exclusion_files_missing"]),
    "disjointness_verified": bool(PROMPT_SUMMARY["disjointness_verified"]),
}
print(json.dumps(PROMPT_SUMMARY, indent=2, ensure_ascii=False))


In [ ]:
# Cell 28
eval_points, origins = [], {}

def _register(vec, origin):
    key = lambda_key(vec, decimals=LAMBDA_DEDUP_DECIMALS)
    if key not in origins:
        origins[key] = []
        eval_points.append(np.asarray(vec, dtype=np.float64))
    origins[key].append(origin)

# Only the Phase-B subset is merged. With USE_FULL_SEARCH_SET = True, Phase A remains
# complete (75 points), while Phase B remains affordable with the 11 named preferences.
lam_df_B = lam_df[lam_df["p_name"].isin(PHASE_B_NAMES)]
assert len(lam_df_B) > 0, "No lambda row belongs to the Phase-B subset."

for _, r in lam_df_B.iterrows():
    _register(r[p_cols].to_numpy(float), f"baseline:{r['p_name']}")
    if bool(r["usable"]) and bool(r["moved"]):
        _register(r[lam_cols].to_numpy(float), f"{r['method']}({r['params']}):{r['p_name']}")

EVAL_POINTS = np.asarray(eval_points, dtype=np.float64)
n_unique = len(EVAL_POINTS)

# Read the prompt count from the prompt file; do not guess it.
if REWARD_PROMPT_PATH.is_file():
    n_prompts = sum(1 for line in REWARD_PROMPT_PATH.read_text(encoding="utf-8").splitlines() if line.strip())
else:
    n_prompts = 80
    print(f"NOTE: {REWARD_PROMPT_PATH.name} does not exist yet; using {n_prompts} Prompts.")

print(f"Eindeutige merge points:   {n_unique}")
print(f"Prompts je Punkt:          {n_prompts}")
print(f"Generierungen gesamt:      {n_unique * n_prompts}")
print(f"Ohne Deduplizierung:      {len(lam_df) * n_prompts}")
print(f"Ersparnis durch Deduplizierung:     {100 * (1 - n_unique / max(len(lam_df), 1)):.1f} %")
hours = n_unique * n_prompts * 2.5 / 3600
print(f"\nRough runtime (A100, 256 new tokens, ~2.5 s per generation + scoring): ~{hours:.1f} h")
print("  " + ("FITS in one session" if hours < 3 else
              "TOO LONG for one session — reduce the preference set or grid"))

In [ ]:
# Cell 29
print("Phase B remains disabled by default.")
print("After inspecting Phase A, set PREREG_CONFIRM=True and run through the gate.")
print("Only then set RUN_REWARD_COLLECTION=True and rerun from the settings cell.")


**Cell 30**

## 9. Gate — neue Vorregistrierung für das DPO-Regime

Der Gate-Hash bindet Adapter, Manifeste, DPO-Matrizen, Promptdatei,
NB11-Sperrliste, Modellrevisionen, Generierungsparameter, Softwarestände und
die vollständige Koeffiziententabelle. Ein NB10-Cache kann dadurch nicht
versehentlich fortgesetzt werden.


**Cell 31**

### 9.1 Bindungsnachweis

Die Basis- und Datensatzrevision stammen aus den verifizierten
Trainingsmanifesten. ArmoRM wird ebenfalls über einen festen Commit geladen,
nicht über einen veränderlichen Branch-Namen.


In [ ]:
# Cell 32
def _runtime_versions():
    import platform
    from importlib import metadata

    packages = (
        "torch", "transformers", "tokenizers", "peft", "accelerate",
        "datasets", "huggingface_hub", "bitsandbytes", "numpy", "pandas", "scipy",
    )
    return {
        "python": platform.python_version(),
        **{name: metadata.version(name) for name in packages},
    }

def _protocol_notebook_sha256(path):
    """Hash NB13 while treating the two activation switches as runtime controls."""
    text = Path(path).read_text(encoding="utf-8")
    text = re.sub(
        r"(PREREG_CONFIRM|RUN_REWARD_COLLECTION) = (True|False)",
        r"\1 = <ACTIVATION_FLAG>",
        text,
    )
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

BINDING = {
    "schema_version": 1,
    "notebook": "NB13 HelpSteer2 DPO method comparison",
    "claim_status": CLAIM_STATUS,
    "regime": REGIME,
    "adapter_bundle_sha256": ADAPTER_BUNDLE_SHA256,
    "adapter_root": str(ADAPTER_INPUT_ROOT.relative_to(PROJECT_ROOT)),
    "adapter_weights_sha256": {
        axis: MANIFESTS[axis]["adapter_model_sha256"] for axis in ATTRIBUTES
    },
    "training_manifests_sha256": {
        axis: _sha256_file(MANIFEST_PATHS[axis]) for axis in ATTRIBUTES
    },
    "training_binding_sha256": {
        axis: MANIFESTS[axis]["binding_sha256"] for axis in ATTRIBUTES
    },
    "training_runtime_versions": MANIFESTS[ATTRIBUTES[0]]["runtime_versions"],
    "base_model": {"name": BASE_MODEL_NAME, "revision": BASE_REVISION},
    "dataset": {"name": DATASET_NAME, "revision": DATASET_REVISION,
                "training_split": "train", "evaluation_split": "validation"},
    "reward_model": {"name": ARMORM_MODEL, "revision": ARMORM_REVISION,
                     "precision": ARMORM_PRECISION, "batch_size": 1},
    "geometry": {
        "primary_matrix": PRIMARY_MATRIX,
        "R_cos_sha256": _sha256_file(COSINE_MATRIX_CSV),
        "R_gram_sha256": _sha256_file(GRAM_MATRIX_CSV),
        "norms_sha256": _sha256_file(D_NORMS_CSV),
        "report_sha256": _sha256_file(GEOMETRY_JSON),
    },
    "prompts": {
        "evaluation_sha256": _sha256_file(REWARD_PROMPT_PATH),
        "nb11_exclusion_file_sha256": _sha256_file(DIAGNOSTIC_EXCLUSION_PATH),
        "nb11_prompt_list_sha256": NB11_DIAGNOSTIC_PROMPT_SHA256,
        "selection": PROMPT_BINDING,
    },
    "attribute_order": list(ATTRIBUTES),
    "generation": {
        "max_new_tokens": MAX_NEW_TOKENS,
        "repetition_penalty": REPETITION_PENALTY,
        "no_repeat_ngram_size": NO_REPEAT_NGRAM_SIZE,
        "decoding": "greedy; do_sample=False; num_beams=1",
        "prompt_format": "base tokenizer chat template; add_generation_prompt=True",
        "merge_dtype": "float32",
    },
    "grids": {
        "rho_grid": RHO_GRID, "cert_c": CERT_C, "cert_eps": CERT_EPS,
        "c_grid": C_GRID, "alpha_grid": ALPHA_GRID, "eps_grid": EPS_GRID,
    },
    "source_sha256": {
        name: _sha256_file(PROJECT_ROOT / "src" / name)
        for name in (
            "coefficient_portfolio.py", "merge.py", "proxy_validation.py",
            "metrics.py", "lambda_utils.py", "armorm_scorer.py",
            "armorm_objectives.py", "eval_prompts.py", "effective_lora_geometry.py",
        )
    },
    "protocol_notebook_sha256": _protocol_notebook_sha256(NOTEBOOK_PATH),
    "evaluation_runtime_versions": _runtime_versions(),
}
BINDING_SHA256 = hashlib.sha256(
    json.dumps(BINDING, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()
print(json.dumps(BINDING, indent=2))
print(f"BINDING_SHA256 = {BINDING_SHA256}")


In [ ]:
# Cell 33
lambda_hash = hashlib.sha256(
    lam_df[["p_name", "method", "params", "status"] + lam_cols]
    .round(9).to_csv(index=False).encode("utf-8")
).hexdigest()
floor_summary = {
    "n_preferences": len(floor_by_preference),
    "n_collapsed": sum(entry["collapsed"] for entry in floor_by_preference.values()),
    "per_preference": floor_by_preference,
}

prereg = {
    "schema_version": 1,
    "run_tag": RUN_TAG,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "NB13 HelpSteer2 DPO method comparison",
    "claim_status": CLAIM_STATUS,
    "lambda_table_sha256": lambda_hash,
    "binding": BINDING,
    "binding_sha256": BINDING_SHA256,
    "primary_matrix": PRIMARY_MATRIX,
    "floor_lp_summary": floor_summary,
    "methods": {
        "Baseline": {"definition": "lambda=p"},
        "Avg": {"source": "src.coefficient_portfolio.avg", "rho_grid": RHO_GRID},
        "Cert": {
            "source": "src.coefficient_portfolio.cert", "c": CERT_C, "eps": CERT_EPS,
            "note": "computed anew on R_DPO; no NB09/NB10 certificate is reused",
        },
        "MaxMin": {"source": "src.coefficient_portfolio.maxmin_c", "c_grid": C_GRID},
        "Fair": {"source": "src.coefficient_portfolio.fair_alpha_eps",
                 "alpha_grid": ALPHA_GRID, "eps_grid": EPS_GRID},
    },
    "n_preferences_phase_a": len(PREF_SET),
    "n_preferences_phase_b": len(PREF_SET_B),
    "phase_b_preference_rule": PHASE_B_RULE,
    "phase_b_preference_names": [name for name, _ in PREF_SET_B],
    "n_unique_merge_points": int(n_unique),
    "metric_primary": "raw U_p = sum_i p_i r_i using five anchored ArmoRM heads",
    "metric_sensitivity": ["minmax", "rank"],
    "proxy_diagnostics": (
        "descriptive Spearman analysis for R_cos/R_gram geometry scores and "
        "predicted-vs-observed method deltas; never used to select a grid point"
    ),
    "error_layer": "paired bootstrap over prompts; 10000 draws; alpha=0.05",
    "multiplicity": "Holm correction over the complete NB13 method/preference family",
    "decision_rule": (
        "Report Delta U_p, paired percentile CI, raw p and Holm-adjusted p for every "
        "usable moving Phase-B row. Improvement requires Delta U_p>0 and p_Holm<0.05; "
        "harm is analogous. Because CLAIM_STATUS is exploratory by default, these are "
        "exploratory multiplicity-controlled findings unless status was changed and "
        "frozen before any Phase-B reward contact."
    ),
    "armorm_role": (
        "ArmoRM is post-hoc only. NB11 manifests prove it was absent from pair selection, "
        "DPO optimization and checkpoint selection. NB11's 64 diagnostic prompts are "
        "reconstructed by their frozen list hash and excluded from NB13."
    ),
    "prior_results_policy": (
        "No NB11 diagnostic rewards, NB10 rewards, matrices, certificates or caches are "
        "used to choose NB13 coefficients or prompts."
    ),
    "raw_scores_retained": True,
}

if PREREG_JSON.exists():
    frozen = json.loads(PREREG_JSON.read_text(encoding="utf-8"))
    print(f"Preregistration already exists ({frozen['created_utc']}); not overwritten.")
    if frozen.get("lambda_table_sha256") != lambda_hash or frozen.get("binding_sha256") != BINDING_SHA256:
        print("*** WARNING: current run differs from the frozen preregistration. ***")
elif PREREG_CONFIRM:
    write_json(PREREG_JSON, prereg)
    print(f"Preregistration frozen -> {PREREG_JSON}")
else:
    print("PREREG_CONFIRM=False — preview only; Phase B remains closed.")
    print(json.dumps(prereg, indent=2)[:2500])


In [ ]:
# Cell 34
GATE_OPEN = False
if PREREG_JSON.exists():
    frozen = json.loads(PREREG_JSON.read_text(encoding="utf-8"))
    differences = []
    if frozen.get("lambda_table_sha256") != lambda_hash:
        differences.append("lambda_table_sha256")
    if frozen.get("binding_sha256") != BINDING_SHA256:
        differences.append("binding_sha256")
    GATE_OPEN = not differences
    print("GATE OPEN" if GATE_OPEN else "GATE CLOSED — mismatch: " + ", ".join(differences))
else:
    print("GATE CLOSED — no NB13 preregistration.")

if RUN_REWARD_COLLECTION and not GATE_OPEN:
    raise RuntimeError(
        "RUN_REWARD_COLLECTION=True but the NB13 DPO binding is not frozen and matching."
    )


**Cell 35**

## 10. Phase B — post-hoc Reward-Evaluation

### 10.1 Exaktes Mergen und Generation

Die DPO-Updates werden relativ zum exakt gepinnten, unveränderten
TinyLlama-Commit in float32 addiert und nach jedem Merge vollständig
zurückgesetzt. PEFTs lineare Faktoraddition wird nicht verwendet, weil sie
bei inneren Koeffizienten unerwünschte Kreuzterme erzeugen kann.


In [ ]:
# Cell 36
if RUN_REWARD_COLLECTION:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from src.merge import combine_effective_deltas, effective_deltas, resolve_base_module

    assert BINDING["adapter_weights_sha256"] == {
        axis: _sha256_file(ADAPTER_PATHS[axis] / "adapter_model.safetensors")
        for axis in ATTRIBUTES
    }
    generation_tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL_NAME, revision=BASE_REVISION, use_fast=True
    )
    generation_tokenizer.pad_token = generation_tokenizer.eos_token
    generation_tokenizer.padding_side = "left"
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        revision=BASE_REVISION,
        torch_dtype=torch.float32,
        device_map="auto",
    ).eval()
    DELTAS = effective_deltas(dict(ADAPTER_PATHS))
    print(f"Pinned base loaded; effective DPO deltas prepared for {len(next(iter(DELTAS.values())))} modules.")
else:
    print("RUN_REWARD_COLLECTION=False — Phase B skipped.")


In [ ]:
# Cell 37
if RUN_REWARD_COLLECTION:
    @contextmanager
    def merged_model(model, deltas_by_adapter, lam):
        merged = combine_effective_deltas(lam, deltas_by_adapter)
        originals = {}
        try:
            for module_name, delta_cpu in merged.items():
                module = resolve_base_module(model, module_name)
                originals[module_name] = module.weight.detach().clone()
                update = module.weight.detach().float() + delta_cpu.to(
                    device=module.weight.device, dtype=torch.float32
                )
                with torch.no_grad():
                    module.weight.copy_(update.to(dtype=module.weight.dtype))
            yield model
        finally:
            for module_name, original in originals.items():
                with torch.no_grad():
                    resolve_base_module(model, module_name).weight.copy_(original)
            del originals, merged
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    def generate_answer(prompt: str) -> str:
        device = next(base_model.parameters()).device
        rendered = generation_tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
        encoded = generation_tokenizer(
            rendered,
            return_tensors="pt",
            add_special_tokens=False,
            truncation=True,
            max_length=512,
        )
        input_ids = encoded["input_ids"].to(device)
        with torch.inference_mode():
            generated = base_model.generate(
                input_ids=input_ids,
                attention_mask=encoded["attention_mask"].to(device),
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                num_beams=1,
                repetition_penalty=REPETITION_PENALTY,
                no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
                pad_token_id=generation_tokenizer.pad_token_id,
                eos_token_id=generation_tokenizer.eos_token_id,
            )
        answer_ids = generated[0, input_ids.shape[1]:]
        return generation_tokenizer.decode(answer_ids, skip_special_tokens=True).strip()

    probe = generation_tokenizer.apply_chat_template(
        [{"role": "user", "content": "probe"}], tokenize=False,
        add_generation_prompt=True,
    )
    probe_ids = generation_tokenizer(probe, add_special_tokens=False)["input_ids"]
    bos = generation_tokenizer.bos_token_id
    assert bos is None or probe_ids.count(bos) <= 1
    print("Merge and pinned generation path are ready:", repr(probe))


In [ ]:
# Cell 38
if RUN_REWARD_COLLECTION:
    import logging
    import os
    import shutil
    import time
    import traceback
    from datetime import timedelta
    from pathlib import Path
    from zoneinfo import ZoneInfo
    from src.armorm_objectives import ARMORM_HELPSTEER_OBJECTIVE_NAMES
    from src.armorm_scorer import make_score_prompt_answer
    from src.proxy_validation import collect_reward_tensor
    from src.tinyllama_training_utils import load_reward_prompts
    _cell38_started = time.perf_counter()

    # Colab can terminate the VM without a Python exception. The cache and diagnostics are
    # therefore stored on Google Drive. If the last event has no completion/exception,
    # the failure occurred outside Python (host reclaim, hard process kill, etc.).
    _local_reward_cache = Path(REWARD_CACHE)
    DIAGNOSTIC_LOG = RESULTS_DIR / "nb13_runtime_diagnostics.jsonl"
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        _drive_root = Path("/content/drive/MyDrive")
        if not _drive_root.is_dir():
            raise RuntimeError("Google Drive was not mounted at /content/drive/MyDrive.")
        _persistent_dir = _drive_root / "master-thesis-nb13" / RUN_TAG
        _persistent_dir.mkdir(parents=True, exist_ok=True)
        _persistent_cache = _persistent_dir / "reward_cache.jsonl"
        if (_local_reward_cache.is_file() and not _persistent_cache.exists()
                and _local_reward_cache.stat().st_size > 0):
            shutil.copy2(_local_reward_cache, _persistent_cache)
        REWARD_CACHE = _persistent_cache
        DIAGNOSTIC_LOG = _persistent_dir / "runtime_diagnostics.jsonl"
        print(f"[persistent] Reward cache: {REWARD_CACHE}")
        print(f"[persistent] Diagnostics:    {DIAGNOSTIC_LOG}")
    except Exception as _drive_error:
        print("[WARNING] Google Drive persistence is unavailable; /content may be lost after "
              f"a runtime failure: {_drive_error}")

    def _resource_snapshot():
        snapshot = {}
        try:
            import psutil

            vm = psutil.virtual_memory()
            snapshot.update({
                "ram_used_gib": round((vm.total - vm.available) / 2**30, 3),
                "ram_total_gib": round(vm.total / 2**30, 3),
                "ram_percent": float(vm.percent),
            })
        except Exception as error:
            snapshot["ram_error"] = f"{type(error).__name__}: {error}"
        try:
            if torch.cuda.is_available():
                free, total = torch.cuda.mem_get_info()
                snapshot.update({
                    "gpu_name": torch.cuda.get_device_name(),
                    "vram_used_gib": round((total - free) / 2**30, 3),
                    "vram_total_gib": round(total / 2**30, 3),
                    "torch_allocated_gib": round(torch.cuda.memory_allocated() / 2**30, 3),
                    "torch_reserved_gib": round(torch.cuda.memory_reserved() / 2**30, 3),
                })
        except Exception as error:
            snapshot["vram_error"] = f"{type(error).__name__}: {error}"
        return snapshot

    def _diagnostic_event(event, **details):
        record = {
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "event": event,
            "run_tag": RUN_TAG,
            "binding_sha256": BINDING_SHA256,
            "resources": _resource_snapshot(),
            **details,
        }
        try:
            DIAGNOSTIC_LOG.parent.mkdir(parents=True, exist_ok=True)
            with DIAGNOSTIC_LOG.open("a", encoding="utf-8") as handle:
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")
                handle.flush()
                os.fsync(handle.fileno())
        except Exception as error:
            print(f"[WARNING] Diagnostic event could not be saved: {error}")

    _previous_event = None
    if DIAGNOSTIC_LOG.is_file():
        for _line in reversed(DIAGNOSTIC_LOG.read_text(encoding="utf-8").splitlines()):
            try:
                _previous_event = json.loads(_line)
                break
            except (json.JSONDecodeError, TypeError):
                continue
    if _previous_event and _previous_event.get("event") != "run_completed":
        print("[diagnose] The previous run ended at:",
              json.dumps(_previous_event, ensure_ascii=False))
        if _previous_event.get("event") not in {"point_exception", "run_exception"}:
            print("[diagnostics] No Python exception was recorded. This suggests "
                  "an externally terminated Colab VM or a hard process kill.")
    _diagnostic_event("cell_started")

    # bitsandbytes emits the same known BF16-to-FP16 conversion message for many
    # layers. Filter only this message; all other warnings remain visible.
    class _BnbCastMessageFilter(logging.Filter):
        def filter(self, record):
            return ("MatMul8bitLt: inputs will be cast from torch.bfloat16 "
                    "to float16 during quantization") not in record.getMessage()

    _bnb_logger = logging.getLogger("bitsandbytes.autograd._functions")
    if not any(getattr(f, "_nb13_bnb_cast_filter", False) for f in _bnb_logger.filters):
        _bnb_filter = _BnbCastMessageFilter()
        _bnb_filter._nb13_bnb_cast_filter = True
        _bnb_logger.addFilter(_bnb_filter)

    reward_prompts = load_reward_prompts(REWARD_PROMPT_PATH)
    assert len(reward_prompts) == n_prompts, "Prompt count differs from the preregistration."
    assert all(a in ARMORM_HELPSTEER_OBJECTIVE_NAMES for a in ATTRIBUTES), \
        "ATTRIBUTES contains an axis without an anchored ArmoRM head mapping."

    # 8-bit matches NB11's post-hoc diagnostic precision. It was never a DPO
    # training signal; the fixed commit and golden sample anchor the evaluator.
    score_prompt_answer, scorer = make_score_prompt_answer(
        ARMORM_MODEL, revision=ARMORM_REVISION, dtype="bfloat16",
        load_in_8bit=(ARMORM_PRECISION == "8bit"))
    assert scorer.describe()["precision"] == ("int8" if ARMORM_PRECISION == "8bit" else "bfloat16")
    scorer.assert_golden_sample()          # BEFORE the first real scoring operation
    print("Scorer:", scorer.describe())
    _diagnostic_event("scorer_ready", scorer=scorer.describe())
    print(f"[time] ArmoRM setup including golden test: "
          f"{time.perf_counter() - _cell38_started:.1f} s")

    def _format_duration(seconds):
        seconds = max(0, int(round(seconds)))
        hours, remainder = divmod(seconds, 3600)
        minutes, seconds = divmod(remainder, 60)
        return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

    # Time only uncached merge points. Mapping via
    # coefficient_key remains correct even after an interrupted run.
    _cached_keys = set()
    if REWARD_CACHE.is_file():
        for _line in REWARD_CACHE.read_text(encoding="utf-8").splitlines():
            if not _line.strip():
                continue
            try:
                _record = json.loads(_line)
            except json.JSONDecodeError:
                continue
            if "key" in _record:
                _cached_keys.add(str(_record["key"]))
    _eval_index_by_key = {coefficient_key(lam): i + 1
                          for i, lam in enumerate(EVAL_POINTS)}
    _cached_eval_count = sum(key in _cached_keys for key in _eval_index_by_key)
    _missing_initial = n_unique - _cached_eval_count
    _progress = {"started": time.perf_counter(), "new_done": 0}
    _nominal_remaining = _missing_initial * n_prompts * 2.5
    print(f"[time] Cell 38: {n_unique} merge points x {n_prompts} Prompts; "
          f"{_cached_eval_count} from cache, {_missing_initial} remaining.")
    print(f"[time] Initial remaining-time estimate at 2.5 s per prompt pair: "
          f"{_format_duration(_nominal_remaining)}.")

    def reward_of_lambda(lam):
        """Per-prompt ArmoRM rewards for one merge point, shape (n_prompts, m)."""
        _point_started = time.perf_counter()
        _key = coefficient_key(lam)
        _position = _eval_index_by_key[_key]
        _new_number = _progress["new_done"] + 1
        print(f"[time] Starting merge point {_position}/{n_unique} "
              f"(remaining point {_new_number}/{_missing_initial}).")
        _diagnostic_event("point_started", position=_position, total=n_unique,
                          open_number=_new_number, open_total=_missing_initial,
                          coefficient_key=_key, coefficients=np.asarray(lam).tolist())
        try:
            with merged_model(base_model, DELTAS, lam):
                answers = [generate_answer(record["prompt"]) for record in reward_prompts]
            _diagnostic_event("generation_finished", position=_position,
                              answers=len(answers))
            scores = np.asarray([
                score_prompt_answer(record["prompt"], answer, ATTRIBUTES)
                for record, answer in zip(reward_prompts, answers)
            ], dtype=np.float64)
            assert scores.shape == (len(reward_prompts), m), scores.shape
            _progress["new_done"] += 1
            _point_elapsed = time.perf_counter() - _point_started
            _reward_elapsed = time.perf_counter() - _progress["started"]
            _cell_elapsed = time.perf_counter() - _cell38_started
            _average = _reward_elapsed / _progress["new_done"]
            _remaining = _missing_initial - _progress["new_done"]
            _eta_seconds = _average * _remaining
            _finish = (datetime.now(ZoneInfo("Europe/Berlin")) + timedelta(seconds=_eta_seconds)).strftime("%d.%m. %H:%M %Z")
            _diagnostic_event("point_scoring_finished", position=_position,
                              elapsed_seconds=_point_elapsed,
                              mean_reward=scores.mean(axis=0).tolist())
            print(f"[time] Completed {_progress['new_done']}/{_missing_initial} remaining | "
                  f"last point {_format_duration(_point_elapsed)} | "
                  f"reward time {_format_duration(_reward_elapsed)} | "
                  f"cell time {_format_duration(_cell_elapsed)} | "
                  f"remaining {_format_duration(_eta_seconds)} | expected {_finish}")
            return scores      # Do NOT average: the paired bootstrap requires raw values
        except BaseException as error:
            _diagnostic_event("point_exception", position=_position,
                              error_type=type(error).__name__, error=str(error),
                              traceback=traceback.format_exc())
            raise

    try:
        REWARD_TENSOR = collect_reward_tensor(
            EVAL_POINTS, reward_of_lambda, REWARD_CACHE,
            num_prompts=n_prompts, binding_sha256=BINDING_SHA256)
        REWARD_MATRIX = REWARD_TENSOR.mean(axis=1)  # identical to the earlier matrix
        np.save(REWARD_TENSOR_PATH, REWARD_TENSOR)
        _diagnostic_event("run_completed", reward_tensor_shape=list(REWARD_TENSOR.shape))
        print(f"Reward tensor: {REWARD_TENSOR.shape}   Reward matrix: {REWARD_MATRIX.shape}")
    except BaseException as error:
        _diagnostic_event("run_exception", error_type=type(error).__name__,
                          error=str(error), traceback=traceback.format_exc())
        print(f"[diagnostics] Failure reason saved persistently to {DIAGNOSTIC_LOG}")
        raise


**Cell 39**

## 11. Ergebnistabelle

Die Tabelle enthält nur die 11 vorab festgelegten Phase-B-Präferenzen.
Kollabierte Methoden teilen denselben Reward-Cacheeintrag wie $\lambda=p$;
ein nichttriviales Cert-Ergebnis wird dagegen wie jeder andere Mergepunkt
tatsächlich ausgewertet.


In [ ]:
# Cell 40
def build_final_table() -> pd.DataFrame:
    reward_by_key = {}
    if REWARD_CACHE.is_file() and REWARD_CACHE.stat().st_size > 0:
        for line in REWARD_CACHE.read_text(encoding="utf-8").splitlines():
            if line.strip():
                record = json.loads(line)
                if "key" in record and "reward" in record:
                    reward_by_key[str(record["key"])] = np.asarray(record["reward"], dtype=np.float64)

    output = []
    for _, row in lam_df_B.iterrows():
        p = row[p_cols].to_numpy(float)
        usable = bool(row["usable"])
        lam = row[lam_cols].to_numpy(float) if usable else None
        result = {
            "p_name": row["p_name"], "p": np.round(p, 6).tolist(),
            "method": row["method"], "params": row["params"], "status": row["status"],
            "lambda": np.round(lam, 6).tolist() if usable else None,
            "dist_l2": row["dist_l2"], "dist_R": row["dist_R"],
            "proxy_delta_p": (float(p @ R @ (lam - p)) if usable else np.nan),
            "proxy_min_delta": row["min_delta"],
        }
        baseline_reward = reward_by_key.get(coefficient_key(p))
        lambda_reward = reward_by_key.get(coefficient_key(lam)) if usable else None
        if baseline_reward is not None:
            u_base = float(baseline_reward @ p)
            u_lambda = float(lambda_reward @ p) if lambda_reward is not None else np.nan
            note = "same merge point as baseline" if usable and np.linalg.norm(lam - p) <= 1e-8 else ""
        else:
            u_base = u_lambda = np.nan
            note = "Phase B was not run" if not reward_by_key else "reward missing"
        result.update({
            "U_p(p)": u_base,
            "U_p(lambda)": u_lambda,
            "Delta U_p": u_lambda - u_base,
            "note": row["status"] if not usable else note,
        })
        output.append(result)
    return pd.DataFrame(output)

final_df = build_final_table()
final_df.to_csv(FINAL_CSV, index=False)
print(f"Final table -> {FINAL_CSV} ({len(final_df)} rows)")
with pd.option_context("display.max_rows", 250, "display.width", 240):
    display(final_df.round(5))


**Cell 41**

### 11.1 Verdichtete Sicht nach Methode

Leere Rewardfelder bedeuten, dass Phase B nicht lief oder ein Punkt
unzulässig war; sie werden nicht als Null-Effekt interpretiert.


In [ ]:
# Cell 42
agg = {"n": ("dist_l2", "size"),
       "n_usable": ("lambda", lambda s: int(s.notna().sum())),
       "mean_dist_l2": ("dist_l2", "mean"),
       "mean_dist_R": ("dist_R", "mean")}
if final_df["Delta U_p"].notna().any():
    agg.update({"mean_Delta_U_p": ("Delta U_p", "mean"),
                "n_improved": ("Delta U_p", lambda s: int((s > 0).sum())),
                "n_worse": ("Delta U_p", lambda s: int((s < 0).sum()))})
grouped = final_df.assign(
    key=final_df["method"] + final_df["params"].map(lambda s: f"({s})" if s else "")
).groupby("key").agg(**agg)
display(grouped.round(5))

**Cell 43**

## 12. Robustheit gegenüber Achsenskalierung

Der rohe präferenzgewichtete Score bleibt primär. Rang- und Min-Max-Skalierung
werden auf derselben Rewardmatrix als Sensitivitätsanalyse ausgewiesen; sie
verursachen keinen zusätzlichen Modellkontakt.


In [ ]:
# Cell 44
if REWARD_CACHE.is_file() and REWARD_CACHE.stat().st_size > 0:
    _cache_records = [json.loads(line) for line in
                      REWARD_CACHE.read_text(encoding="utf-8").splitlines() if line.strip()]
    matrix = np.asarray([record["reward"] for record in _cache_records
                         if "reward" in record], dtype=np.float64)

    robust_rows = []
    for pname, p in PREF_SET_B:
        agreement = normalization_agreement(matrix, p)
        robust_rows.append({
            "p_name": pname,
            "argmax_agrees": agreement["argmax_agrees"],
            **{f"argmax_{k}": v for k, v in agreement["argmax_index"].items()},
            **{f"rho_{k}": v for k, v in agreement["spearman"].items()},
        })
    robust_df = pd.DataFrame(robust_rows)
    robust_df.to_csv(ROBUST_CSV, index=False)
    display(robust_df.round(4))
    n_agree = int(robust_df["argmax_agrees"].sum())
    print(f"\nargmax agrees for {n_agree}/{len(robust_df)} preferences across all three normalizations.")
    print("If it agrees throughout, the scale objection to the raw sum is answered empirically;")
    print("if it differs, this must be reported as a limitation, not used as a reason to switch metrics.")
else:
    print("No reward cache — robustness comparison skipped.")

**Cell 45**

## 13. Gepaarte Unsicherheit und Multiplizität

Für jeden bewegten Kandidaten wird die promptweise Differenz

$$d_j=p^\top r_j(\lambda)-p^\top r_j(p)$$

gebootstrapped. Holm kontrolliert die vollständige NB13-Familie. Bei
`CLAIM_STATUS="exploratory"` bleiben auch Holm-signifikante Befunde
explorativ; der Status darf nach Rewardkontakt nicht geändert werden.


In [ ]:
# Cell 46
if RUN_REWARD_COLLECTION:
    from src.lambda_utils import holm_adjust, lambda_key
    from src.metrics import mean_rank, paired_bootstrap_ci, selection_regret

    # Use the same key function as the deduplication in Section 6c; otherwise
    # the lookup will not find the points again.
    def _key(vec):
        return lambda_key(np.asarray(vec, dtype=float), decimals=LAMBDA_DEDUP_DECIMALS)

    point_index = {_key(pt): i for i, pt in enumerate(EVAL_POINTS)}
    stats_rows, utilities_by_method = [], {}

    for _, r in lam_df_B.iterrows():
        if not (bool(r["usable"]) and bool(r["moved"])):
            continue
        p_vec = r[p_cols].to_numpy(float)
        lam_vec = r[lam_cols].to_numpy(float)
        i_lam, i_p = point_index.get(_key(lam_vec)), point_index.get(_key(p_vec))
        if i_lam is None or i_p is None:
            raise KeyError(f"Merge point is missing from the tensor: {r['p_name']}/{r['method']}")

        boot = paired_bootstrap_ci(REWARD_TENSOR[i_lam], REWARD_TENSOR[i_p], p_vec)
        label = f"{r['method']}({r['params']})" if r["params"] else str(r["method"])
        stats_rows.append({
            "p_name": r["p_name"], "method": r["method"], "params": r["params"], "label": label,
            "U_p_baseline": float(REWARD_TENSOR[i_p].mean(axis=0) @ p_vec),
            "U_p_lambda": float(REWARD_TENSOR[i_lam].mean(axis=0) @ p_vec),
            **{k: boot[k] for k in ("delta_u_p", "ci_low", "ci_high", "excludes_zero", "p_value")},
        })
        utilities_by_method.setdefault(label, {})[r["p_name"]] = stats_rows[-1]["U_p_lambda"]
        utilities_by_method.setdefault("Baseline (lambda=p)", {})[r["p_name"]] = \
            stats_rows[-1]["U_p_baseline"]

    stats_df = pd.DataFrame(stats_rows)
    if len(stats_df):
        # ONLY Holm is confirmatory. The unadjusted CI remains as a
        # descriptive column but carries no claim: with approximately two hundred
        # Vergleichen erzeugt alpha = 0.05 etwa zehn Zufallstreffer.
        stats_df["p_holm"] = holm_adjust(stats_df["p_value"].to_numpy(float))
        stats_df["holm_improves"] = (stats_df["delta_u_p"] > 0) & (stats_df["p_holm"] < 0.05)
        stats_df["holm_harms"] = (stats_df["delta_u_p"] < 0) & (stats_df["p_holm"] < 0.05)
        stats_df["significant"] = stats_df["holm_improves"] | stats_df["holm_harms"]

        common = set.intersection(*(set(v) for v in utilities_by_method.values()))
        if common:
            order = sorted(common)
            ranks = mean_rank({k: [v[n] for n in order] for k, v in utilities_by_method.items()})
            print(f"Mean rank across {len(order)} preferences (1 = best):")
            for name, value in sorted(ranks.items(), key=lambda kv: kv[1]):
                print(f"  {value:5.2f}  {name}")

        # Selection regret against the best point in the ENTIRE evaluated set, not
        # only against points generated for this preference. This is the only way to keep the reference set
        # identical for all methods and make the value match the wording
        # "best evaluated point" in the thesis.
        for pname, p_vec in PREF_SET_B:
            search_utilities = (REWARD_MATRIX @ np.asarray(p_vec, dtype=float)).tolist()
            mask = stats_df["p_name"] == pname
            if not bool(mask.any()):
                continue  # no moving method for this preference
            stats_df.loc[mask, "selection_regret"] = [
                selection_regret(u, search_utilities) for u in stats_df.loc[mask, "U_p_lambda"]]
            stats_df.loc[mask, "baseline_regret"] = selection_regret(
                float(stats_df.loc[mask, "U_p_baseline"].iloc[0]), search_utilities)

        stats_df.to_csv(STATS_CSV, index=False)
        print(f"\n{len(stats_df)} comparisons, "
              f"{int(stats_df['excludes_zero'].sum())} with a CI excluding zero, "
              f"{int(stats_df['holm_improves'].sum())} better after Holm, "
              f"{int(stats_df['holm_harms'].sum())} worse after Holm.")
        display(stats_df.sort_values("delta_u_p", ascending=False).round(5))
    else:
        print("No moving lambda vectors in the Phase-B subset; "
              "all usable method points coincide with their baselines.")
else:
    print("RUN_REWARD_COLLECTION is False — statistics skipped.")


**Cell 47**

## 14. Deskriptive Proxyvalidierung

Diese Auswertung beantwortet getrennt vom Methodenranking, ob die
DPO-Geometrie mit den beobachteten ArmoRM-Unterschieden zusammenhängt.
Sie berichtet (a) die bestehende Spearman-Analyse über alle evaluierten
Mergepunkte und (b) vorhergesagte gegen beobachtete $\Delta U_p$ für die
tatsächlich bewegten Methodenzeilen. Die Ergebnisse dürfen nicht zur
nachträglichen Grid-Auswahl verwendet werden.


In [ ]:
# Cell 48
PROXY_REPORT = None
if RUN_REWARD_COLLECTION:
    PROXY_REPORT = run_spearman_analysis(
        EVAL_POINTS,
        REWARD_MATRIX,
        R_cos,
        R_gram,
        {name: p for name, p in PREF_SET_B},
    )

    proxy_rows = []
    for _, row in lam_df_B.iterrows():
        if not (bool(row["usable"]) and bool(row["moved"])):
            continue
        p = row[p_cols].to_numpy(float)
        lam = row[lam_cols].to_numpy(float)
        i_lam = point_index[_key(lam)]
        i_base = point_index[_key(p)]
        proxy_rows.append({
            "p_name": row["p_name"],
            "method": row["method"],
            "params": row["params"],
            "predicted_delta_u_p": float(p @ R @ (lam - p)),
            "predicted_min_axis_delta": float(np.min(R @ (lam - p))),
            "observed_delta_u_p": float(
                (REWARD_TENSOR[i_lam].mean(axis=0) - REWARD_TENSOR[i_base].mean(axis=0)) @ p
            ),
        })
    proxy_df = pd.DataFrame(proxy_rows)
    proxy_df.to_csv(PROXY_CSV, index=False)

    correlations = {}
    if len(proxy_df):
        for label, group in [("pooled", proxy_df), *proxy_df.groupby("method")]:
            correlations[str(label)] = {
                "n": int(len(group)),
                "spearman_predicted_vs_observed_delta": safe_spearman(
                    group["predicted_delta_u_p"].to_numpy(float),
                    group["observed_delta_u_p"].to_numpy(float),
                ),
            }
    PROXY_REPORT["method_delta_correlations"] = correlations
    PROXY_REPORT["role"] = "descriptive only; not a grid-selection criterion"
    write_json(PROXY_JSON, PROXY_REPORT)
    print(json.dumps(PROXY_REPORT["aggregate"], indent=2))
    display(proxy_df.round(5))
else:
    print("RUN_REWARD_COLLECTION=False — proxy validation skipped.")


**Cell 49**

## 15. Export


In [ ]:
# Cell 50
report = {
    "schema_version": 1,
    "run_tag": RUN_TAG,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "NB13 HelpSteer2 DPO method comparison",
    "claim_status": CLAIM_STATUS,
    "regime": REGIME,
    "primary_matrix": PRIMARY_MATRIX,
    "geometry": GEOMETRY_SUMMARY,
    "adapter_bundle_sha256": ADAPTER_BUNDLE_SHA256,
    "training_method": "DPO",
    "armorm_used_during_training": False,
    "armorm_role": "post-hoc evaluator only",
    "armorm_precision": ARMORM_PRECISION,
    "armorm_revision": ARMORM_REVISION,
    "n_preferences_phase_a": len(PREF_SET),
    "n_preferences_phase_b": len(PREF_SET_B),
    "n_lambda_rows": int(len(lam_df)),
    "n_unique_merge_points": int(n_unique),
    "phase_b_run": bool(RUN_REWARD_COLLECTION),
    "gate_open": bool(GATE_OPEN),
    "lambda_table_sha256": lambda_hash,
    "binding_sha256": BINDING_SHA256,
    "prompt_provenance": PROMPT_BINDING,
    "floor_lp_summary": floor_summary,
    "status_counts": lam_df["status"].value_counts().to_dict(),
    "metric_primary": "raw U_p; identity normalization",
    "proxy_validation_written": bool(PROXY_JSON.is_file()),
}
write_json(REPORT_JSON, report)

zip_path = RESULTS_DIR / f"{RUN_TAG}_outputs.zip"
export_paths = (
    LAMBDA_CSV, FINAL_CSV, ROBUST_CSV, STATS_CSV, PROXY_CSV, PROXY_JSON,
    REPORT_JSON, PREREG_JSON, REWARD_CACHE, REWARD_PROMPT_PATH,
    DIAGNOSTIC_EXCLUSION_PATH, COSINE_MATRIX_CSV, GRAM_MATRIX_CSV,
    D_NORMS_CSV, GEOMETRY_JSON, REWARD_TENSOR_PATH,
)
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in export_paths:
        if Path(path).is_file():
            archive.write(Path(path), Path(path).name)
    if "DIAGNOSTIC_LOG" in globals() and Path(DIAGNOSTIC_LOG).is_file():
        archive.write(Path(DIAGNOSTIC_LOG), "runtime_diagnostics.jsonl")
print(f"{zip_path}\nSHA256 {_sha256_file(zip_path)}")

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    pass


In [ ]:
# Cell 51
!git status --short
